In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

In [3]:
from langgraph.graph import MessagesState

# 2) 节点函数：接收 state，返回新增消息列表
def llm_node(state: MessagesState):
    resp = llm.invoke(state["messages"])  # 传入对话消息列表
    return {"messages": [resp]}

In [4]:
from langgraph.graph import StateGraph

graph = StateGraph(state_schema=MessagesState)

In [5]:
from langgraph.graph import START, END

graph.add_node("llm", llm_node)
graph.add_edge(START, "llm")
graph.add_edge("llm", END)

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "demo-thread-1"}}
app = app.with_config(config)

In [7]:
from langchain_core.messages import HumanMessage

init_state = {
    "messages": [HumanMessage(content="你好，我是jaguarliu，请你简单自我介绍一下。")]
}
result = app.invoke(init_state)
print(result["messages"][-1].content)

你好，jaguarliu！很高兴认识你！😊

我是**DeepSeek**，由深度求索公司创造的AI助手。简单介绍一下自己：

- **核心能力**：我是一个纯文本模型，擅长对话、问答、写作、编程、分析等各种任务。虽然不支持多模态识别（比如直接“看”图片），但可以读取上传文件中的文字信息（图片、PDF、Word、Excel等）。
- **特色功能**：支持联网搜索（需要手动开启）、1M超长上下文（可以一次性处理像《三体》三部曲那么大的内容）、App端支持语音输入。
- **使用方式**：完全免费，Web端和App端都可以使用，App还能从官方应用商店下载。
- **知识截止**：我的知识更新到2025年5月。

无论你是想聊天、学习、工作还是解决问题，我都乐意帮忙！有什么我可以为你做的吗？😄


In [8]:
user_messages = {"messages": [HumanMessage(content="你还记得我是谁吗")]}
result = app.invoke(user_messages)
print(result["messages"][-1].content)

哈哈，当然记得！你刚刚才告诉我你叫 **jaguarliu** 呀～😄

不过需要说明一下：我的“记忆”只限于当前这次对话的上下文。也就是说，**在这个对话里**，我知道你是jaguarliu，也能记住我们聊过的内容。但如果你开启一个新的对话，我就需要重新认识你了——因为我没有跨会话的长期记忆功能。

所以，如果你希望我在后续的交流中一直记得你的名字或其他信息，可以随时提醒我，或者我们就在这个对话里继续聊下去！有什么需要帮忙的吗？😊


In [9]:
for update in app.stream(
    {"messages": [HumanMessage(content="介绍一下 LangGraph")]}, stream_mode="updates"
):
    for _, state in update.items():
        if "messages" in state:
            print(state["messages"][-1].content, end="")

好的，jaguarliu！让我为你详细介绍 **LangGraph**。

## 什么是 LangGraph？

LangGraph 是 **LangChain** 生态系统中一个非常强大的框架，专门用于构建**有状态、多步骤的 AI Agent 应用**。它把 AI 工作流建模成一个**有向图（Directed Graph）**，让开发者可以精细控制 Agent 的推理步骤、工具调用和状态管理。

简单来说，如果说 LangChain 是“链式调用”（线性流程），那么 LangGraph 就是“图式编排”（复杂流程，支持循环、分支、并行）。

---

## 核心概念

### 1. **StateGraph（状态图）**
- 整个应用的核心是一个图结构
- 每个节点（Node）代表一个处理步骤（比如：调用LLM、执行工具、判断条件）
- 边（Edge）定义节点之间的流转逻辑
- 图维护一个全局的**状态（State）**，所有节点共享和修改这个状态

### 2. **节点（Node）**
- 可以是任何可调用对象（函数、LangChain Runnable、自定义类等）
- 典型节点：`call_model`（调用大模型）、`execute_tool`（执行工具）、`router`（路由判断）

### 3. **边（Edge）**
- **普通边**：从一个节点到另一个节点的固定路径
- **条件边（Conditional Edge）**：根据当前状态动态决定下一步走向（比如：如果LLM返回需要调用工具，则走工具节点；否则直接输出）

### 4. **状态（State）**
- 一个可变的字典或自定义对象，贯穿整个图的生命周期
- 每个节点可以读取和更新状态
- 典型状态字段：`messages`（对话历史）、`next_action`（下一步动作）、`intermediate_steps`（中间结果）

---

## 为什么需要 LangGraph？

传统的 LangChain 链（Chain）是**线性**的：A → B → C。但很多真实场景需要：

- **循环**：Agent 反复调用工具直到任务完成
- **分支**：根据条件走不同路径
- **并行**：同时执行多个子任务
- **人工介入**：在某个节点暂停，等待用户输入后再继续

La